In [1]:
import torch 
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 
from torch.utils.data import DataLoader 
import torchvision.datasets as datasets 
import torchvision.transforms as transforms 


In [2]:
# simple nn
class NN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(NN, self).__init__()
        self.fc1 = nn.Linear(input_size, 50)
        self.fc2 = nn.Linear(50, num_classes)
    
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x 


$$
n_{out} = \left\lfloor\frac{n_{in}+2p-k}{s}\right\rfloor +1
$$
$$
n_{in} \text{: number of input features  }\\
n_{out} \text{ : number of output features }\\
\text{k : convolution kernel size }\\
\text{p : convolution padding size }\\
\text{s : convolution stride size 
}
$$

In [8]:
# a simple cnn
class CNN(nn.Module):
    def __init__(self, in_channels =1 , num_classes =10 ):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels= 1, out_channels= 8, kernel_size= (3,3), stride=(1,1), padding=(1,1))  # this is same convolution layer has input and op equal
        self.pool = nn.MaxPool2d(kernel_size=(2,2), stride = (2,2))  #same formula halves the size
        self.conv2 = nn.Conv2d(in_channels= 8, out_channels= 16, kernel_size= (3,3), stride=(1,1), padding=(1,1))  # this is same convolution layer has input and op equal
        self.fc1 = nn.Linear(16*7*7, num_classes)
        
    def forward(self,x):
        x = F.relu(self.conv1(x))
        x = self.pool(x) 
        x = F.relu(self.conv2(x))
        x= self.pool(x) 
        x = x.reshape(x.shape[0], -1)
        x = self.fc1(x)
        
        return x 
    


In [9]:
model = CNN()

x = torch.randn(64,1,28,28)
print(x.shape)
print(model(x).shape)

torch.Size([64, 1, 28, 28])
torch.Size([64, 10])


In [7]:
#set device 
device  =  torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [10]:
#hyper parameters
in_channels = 1 
num_classes = 10 
learning_rate = 0.001
batch_size = 64 
num_epoch = 1

In [11]:
#load data 
train_dataset = datasets.MNIST(root = 'datasets/', train = True, transform = transforms.ToTensor(), download = True)
train_loader = DataLoader(dataset= train_dataset, batch_size= batch_size, shuffle = True)
test_dataset = datasets.MNIST(root = 'datasets/', train = False, transform= transforms.ToTensor(), download = True)
test_loader = DataLoader(dataset= test_dataset, batch_size= batch_size, shuffle = True)

100%|██████████| 9.91M/9.91M [00:02<00:00, 3.78MB/s]
100%|██████████| 28.9k/28.9k [00:02<00:00, 12.2kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 913kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.47MB/s]


In [12]:
#initialize network 
model = CNN().to(device)

#loss and optimizer 
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

In [13]:
#train network 
for epoch in range(num_epoch):
    for batch_idx, (data,targets) in enumerate (train_loader):
        #get data to cuda if possible 
        data = data.to(device = device)
        targets = targets.to(device = device)
        
        #forward 
        scores = model(data)
        loss = criterion(scores, targets)
        
        #backward 
        optimizer.zero_grad()
        loss.backward()
        
        #gradient descent or adam step 
        optimizer.step()


In [14]:
def check_accuracy(loader, model):
    if loader.dataset.train:
        print("checking accuracy on training data")
    else: 
        print("Checking accuracy on test data")
    
    num_correct = 0
    num_samples = 0
    model.eval()
    
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device = device)
            y = y.to(device = device)
            # x = x.reshape(x.shape[0], -1)
            
            scores = model(x)
            _, predictions = scores.max(1)
            num_correct += (predictions == y).sum()
            num_samples += predictions.size(0)
            
        print(f'Got {num_correct}/ {num_samples} with accuracy {float(num_correct)/float(num_samples)*100:.2f}')
        model.train() 


check_accuracy(train_loader, model)
check_accuracy(test_loader, model)




checking accuracy on training data
Got 57957/ 60000 with accuracy 96.59
Checking accuracy on test data
Got 9662/ 10000 with accuracy 96.62
